# 캘리포니아 주택가격 Dataset

**캘리포니아 주택 가격(California Housing) 데이터셋**은 머신러닝에서 회귀(Regression) 문제를 연습할 때 가장 널리 사용되는 대표적인 데이터셋 중 하나입니다.

이 데이터셋은 **1990년 미국 인구조사(Census)** 데이터를 바탕으로 하며, 캘리포니아의 각 조사 구역(Block Group, 대략 600명~3,000명이 거주하는 최소 지리 단위)별 특징과 주택 가격 중간값을 포함하고 있습니다.

### 1. 데이터셋 크기 및 구조 (Shape)

* **샘플 수(행, Instances):** $20,640$개
* **특성 수(열, Attributes):** $8$개의 수치형 독립 변수
* **타깃 변수(Target):** $1$개의 연속형 종속 변수 (주택 가격)
* **결측치(Missing Values):** 존재하지 않음 (모든 데이터가 깔끔하게 채워져 있습니다.)

### 2. 독립 변수 (Features, $8$개)

모델이 주택 가격을 예측하기 위해 사용하는 8개의 입력 데이터 항목입니다.

1. **`MedInc` (Median Income):** 해당 구역 가구들의 **소득 중간값**
* 단위가 만 달러($10,000) 단위로 조정되어 있습니다. (예: `5.0`인 경우 $50,000 달러를 의미)


2. **`HouseAge` (Median House Age):** 해당 구역 주택들의 **연식 중간값** (오래된 정도)
3. **`AveRooms` (Average Rooms):** 가구당 **평균 방 개수**
4. **`AveBedrms` (Average Bedrooms):** 가구당 **평균 침실 개수**
5. **`Population` (Population):** 해당 구역의 **총 인구수**
6. **`AveOccup` (Average Occupancy):** 가구당 **평균 구성원 수**
7. **`Latitude` (Latitude):** 구역의 **위도** (지리적 위치)
8. **`Longitude` (Longitude):** 구역의 **경도** (지리적 위치)


### 3. 종속 변수 (Target, $1$개)

모델이 최종적으로 맞춰야 하는 예측 대상입니다.

* **`MedHouseVal` (Median House Value):** 해당 구역의 **주택 가격 중간값**
* 단위는 10만 달러($100,000)입니다. (예: 타깃 값이 `2.5`라면 해당 구역의 주택 가격 중간값은 $250,000 달러를 의미합니다.)
* 실제 데이터에서는 50만 달러($5.0$) 이상의 주택 가격이 모두 `5.0`으로 상한선(Capping) 처리되어 있는 특징이 있습니다.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

# 1. 데이터 불러오기
california = fetch_california_housing(as_frame=True)

# 2. DataFrame 생성 (상위 5개 행 확인)
df = california.frame
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [2]:
# 요약 정보 확인 (결측치 및 데이터 타입 확인)
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB
None


In [3]:
df.shape

(20640, 9)

In [4]:
df.columns

Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude', 'MedHouseVal'],
      dtype='str')

In [5]:
df.isnull().sum()

MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64

In [6]:
feature_names = [
    "MedInc",
    "HouseAge",
    "AveRooms",
    "AveBedrms",
    "Population",
    "AveOccup",
    "Latitude",
    "Longitude"
]

X = df[feature_names]
y = df["MedHouseVal"]

In [7]:
# train, test 데이터 분리
train_input, test_input, train_target, test_target = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
print("train_input:", train_input.shape)
print("test_input:", test_input.shape)
print("train_target:", train_target.shape)
print("test_target:", test_target.shape)

train_input: (16512, 8)
test_input: (4128, 8)
train_target: (16512,)
test_target: (4128,)


In [9]:
# k-NN 회귀 모델 생성 및 학습
model = KNeighborsRegressor()
model.fit(train_input, train_target)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Uniform weights are used by default.See the following example for a demonstration of the impact ofdifferent weighting schemes on predictions::ref:`sphx_glr_auto_examples_neighbors_plot_regression.py`.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric metric: str, DistanceMetric object or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.If metric is a DistanceMetric object, it will be passed directly tothe underlying computation routines.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [10]:
# 새 데이터 예측
new_data = pd.DataFrame(
    [[8.3252, 41.0, 6.984127, 1.023810, 322.0, 2.555556, 37.88, -122.23]],
    columns=feature_names,
)

pred = model.predict(new_data)

print("예측값:", pred)
print("예측 주택 가격:", pred[0] * 100000, "달러")

예측값: [2.136602]
예측 주택 가격: 213660.19999999995 달러


In [11]:
# 모델 성능 확인
print("train R²:", model.score(train_input, train_target))
print("test R²:", model.score(test_input, test_target))

train R²: 0.45292523357436776
test R²: 0.14631049965900345


In [12]:
# MAE 확인
test_prediction = model.predict(test_input)

mae = mean_absolute_error(test_target, test_prediction)

print("MAE:", mae)
print("평균 오차 금액:", mae * 100000, "달러")

MAE: 0.8127975600775195
평균 오차 금액: 81279.75600775194 달러



---
# 선형 회귀

In [13]:
model = LinearRegression()

model.fit(train_input, train_target)

new_data = pd.DataFrame(
    [[8.3252, 41.0, 6.984127, 1.023810, 322.0, 2.555556, 37.88, -122.23]],
    columns=feature_names,
)

pred = model.predict(new_data)
print("예측 주택 가격:", pred)

model.score(test_input, test_target)

예측 주택 가격: [4.15194306]


0.575787706032451

In [14]:
print("w : ", model.coef_)
print("b : ", model.intercept_)

w :  [ 4.48674910e-01  9.72425752e-03 -1.23323343e-01  7.83144907e-01
 -2.02962058e-06 -3.52631849e-03 -4.19792487e-01 -4.33708065e-01]
b :  -37.02327770606416


In [15]:
print("테스트 데이터 : ", model.score(test_input, test_target))  # 테스트 데이터 평가
print("훈련 데이터 : ", model.score(train_input, train_target))  # 훈련 데이터 평가 

테스트 데이터 :  0.575787706032451
훈련 데이터 :  0.6125511913966952



---
# 다항 회귀

In [16]:
poly = PolynomialFeatures(include_bias=False)  # 절편 포함 X
poly.fit(train_input)  # train에만 fit

# train/test 변환
train_poly = poly.transform(train_input)
test_poly = poly.transform(test_input)  # transform만

In [17]:
# 선형 회귀 모델 학습
model = LinearRegression()
model.fit(train_poly, train_target)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [18]:
# 성능 확인
print("train R²:", model.score(train_poly, train_target))
print("test R²:", model.score(test_poly, test_target))

train R²: 0.6852681982344955
test R²: 0.6456819728455382


In [19]:
# 새 데이터 예측
new_data = pd.DataFrame(
    [[8.3252, 41.0, 6.984127, 1.023810, 322.0, 2.555556, 37.88, -122.23]],
    columns=feature_names,
)

In [20]:
new_poly = poly.transform(new_data)
pred = model.predict(new_poly)

print("예측 주택 가격:", pred)

예측 주택 가격: [4.00752093]
